# Economics lag — do runs react late to margins?

The Economics tab scatter joins **same-week** margins and runs. If teapots / SOEs adjust throughput with a delay, that contemporaneous plot will look like noise even when the lead–lag relationship is real.

This notebook keeps the dashboard untouched and tests:

1. **Single pair deep-dive** — pick one margin model × one runs group, sweep lags
2. **Player comparison** — Independents vs SOEs vs National against each profit model
3. **Complex comparison** — Fuel-oriented vs Chemical vs Aromatics margins as economic signals

Positive lag = **margin leads runs** (economic signal this week → runs move *N* weeks later).

In [12]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fetch_data import load_cached_frames
from src.units import convert_volume_series

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FRAMES = load_cached_frames()
assert FRAMES is not None, "No cache — run: python fetch_data.py --history"

runs_wide = FRAMES["runs_wide"].copy()
margins_wide = FRAMES["margins_wide"].copy()
runs_wide["week_start"] = pd.to_datetime(runs_wide["week_start"])
margins_wide["week_start"] = pd.to_datetime(margins_wide["week_start"])

meta = FRAMES["meta"]
print("Pub date:", meta["pub_date"].iloc[0] if "pub_date" in meta.columns else "?")
print("Runs weeks:", runs_wide["week_start"].min().date(), "→", runs_wide["week_start"].max().date())
print("Margin weeks:", margins_wide["week_start"].min().date(), "→", margins_wide["week_start"].max().date())
print("Runs groups:", sorted(runs_wide["breakdown"].unique()))

Pub date: 2026-07-17 00:00:00
Runs weeks: 2021-01-08 → 2026-07-17
Margin weeks: 2023-01-06 → 2026-07-17
Runs groups: ['Independents', 'National', 'SOEs']


## Config

Change these cells and re-run from here. Defaults mirror the Economics tab screenshot
(Fuel-oriented × net RMB × Independents).

In [13]:
# Deep-dive pair (section below)
FOCUS_REFINERY = "Fuel-oriented refineries"
FOCUS_RUNS = "Independents"
FOCUS_METRIC = "net"          # gross | net | contribution
FOCUS_DISPLAY = "money"       # pct | money  (money ignored for contribution)

# Comparison grids
REFINERY_TYPES = [
    "Fuel-oriented refineries",
    "Oil Refining and Chemical Complexes",
    "Oil Refining and Aromatics Complexes",
]
RUNS_BREAKDOWNS = ["Independents", "SOEs", "National"]
COMPARE_METRIC = "net"
COMPARE_DISPLAY = "money"

# Lag sweep: positive = margin leads runs
MAX_LAG_WEEKS = 12
MIN_OVERLAP = 30  # drop pairs with too few aligned weeks after lag

SHORT = {
    "Fuel-oriented refineries": "Fuel",
    "Oil Refining and Chemical Complexes": "Chem",
    "Oil Refining and Aromatics Complexes": "Aromatics",
}

## Helpers

Column resolution matches `app.py` (`_margin_column`) so notebook and dashboard stay aligned.

In [14]:
def margin_column(
    margins: pd.DataFrame,
    refinery_type: str,
    metric: str,
    display_mode: str = "pct",
) -> str | None:
    """Resolve wide margin column the same way the dashboard does."""
    prefix = refinery_type.replace(" ", "_").replace("-", "")
    if metric == "contribution":
        suffix = "contribution_margin"
    elif metric == "gross":
        suffix = "gross_profit_margin_pct" if display_mode == "pct" else "profit_per_ton"
    elif metric == "net":
        suffix = "net_profit_margin_pct" if display_mode == "pct" else "net_profit"
    else:
        suffix = ""

    for c in margins.columns:
        if c == "week_start":
            continue
        if prefix.lower() not in c.lower():
            continue
        cl = c.lower()
        if suffix == "gross_profit_margin_pct" and "gross_profit_margin" in cl:
            return c
        if suffix == "net_profit_margin_pct" and "net_profit_margin" in cl:
            return c
        if suffix == "profit_per_ton" and "profit_per_ton" in cl:
            return c
        if suffix == "net_profit" and "net_profit" in cl and "margin" not in cl:
            return c
        if suffix == "contribution_margin" and "contribution_margin" in cl:
            return c
    return None


def scale_margin(values: pd.Series, metric: str, display_mode: str) -> pd.Series:
    if metric in ("gross", "net") and display_mode == "pct":
        return values * 100
    return values


def margin_axis_label(metric: str, display_mode: str) -> str:
    if metric == "contribution":
        return "RMB"
    if metric in ("gross", "net") and display_mode == "pct":
        return "%"
    if metric == "gross":
        return "RMB/mt"
    return "RMB"


def aligned_pair(
    runs: pd.DataFrame,
    margins: pd.DataFrame,
    *,
    runs_breakdown: str,
    refinery_type: str,
    metric: str,
    display_mode: str,
) -> pd.DataFrame:
    """One row per overlapping week: runs_kbd + margin (scaled)."""
    col = margin_column(margins, refinery_type, metric, display_mode)
    if col is None:
        return pd.DataFrame(columns=["week_start", "runs_kbd", "margin"])

    run_sub = runs.loc[
        runs["breakdown"] == runs_breakdown, ["week_start", "processing_kt"]
    ].dropna()
    run_sub = run_sub.assign(
        runs_kbd=convert_volume_series(
            run_sub["processing_kt"], "kbd", freq="weekly", metric="processing_kt"
        )
    )

    m = margins[["week_start", col]].dropna().rename(columns={col: "margin_raw"})
    m["margin"] = scale_margin(m["margin_raw"], metric, display_mode)

    out = run_sub.merge(m[["week_start", "margin"]], on="week_start", how="inner")
    return out.sort_values("week_start").reset_index(drop=True)


def lag_correlations(df: pd.DataFrame, max_lag: int, min_overlap: int) -> pd.DataFrame:
    """
    Correlation of margin(t) with runs(t + lag).
    lag > 0  → margin leads runs (runs react later)
    lag < 0  → runs lead margin
    """
    rows = []
    # Include a few negative lags so we can see if causality looks flipped
    for lag in range(-min(4, max_lag), max_lag + 1):
        shifted = df.assign(margin_lagged=df["margin"].shift(lag))
        # shift(+k) moves margin forward in index → pair margin[t-k] with runs[t]
        # Equivalent economic read: margin leads by k weeks when lag > 0
        pair = shifted.dropna(subset=["runs_kbd", "margin_lagged"])
        n = len(pair)
        if n < min_overlap:
            rows.append(
                {"lag_weeks": lag, "n": n, "pearson": np.nan, "spearman": np.nan}
            )
            continue
        rows.append(
            {
                "lag_weeks": lag,
                "n": n,
                "pearson": pair["margin_lagged"].corr(pair["runs_kbd"], method="pearson"),
                "spearman": pair["margin_lagged"].corr(pair["runs_kbd"], method="spearman"),
            }
        )
    return pd.DataFrame(rows)


def best_lag_row(lag_df: pd.DataFrame, score: str = "pearson") -> pd.Series:
    """Best lag among non-negative lags (margin leads), by |correlation|."""
    pos = lag_df[lag_df["lag_weeks"] >= 0].dropna(subset=[score])
    if pos.empty:
        return pd.Series(dtype=float)
    return pos.loc[pos[score].abs().idxmax()]


def apply_lag(df: pd.DataFrame, lag_weeks: int) -> pd.DataFrame:
    out = df.copy()
    out["margin_lagged"] = out["margin"].shift(lag_weeks)
    return out.dropna(subset=["runs_kbd", "margin_lagged"]).reset_index(drop=True)


def ols_fit(x: pd.Series, y: pd.Series) -> tuple[float, float, float]:
    """Simple OLS: y = a + b x. Returns slope, intercept, R²."""
    x = x.astype(float)
    y = y.astype(float)
    if len(x) < 3 or x.std() == 0:
        return np.nan, np.nan, np.nan
    b, a = np.polyfit(x, y, 1)
    yhat = a + b * x
    ss_res = ((y - yhat) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot else np.nan
    return float(b), float(a), float(r2)


def sweep_grid(
    runs: pd.DataFrame,
    margins: pd.DataFrame,
    refinery_types: list[str],
    runs_breakdowns: list[str],
    *,
    metric: str,
    display_mode: str,
    max_lag: int,
    min_overlap: int,
) -> pd.DataFrame:
    """Best lag + corr for every margin model × runs group."""
    rows = []
    for refinery in refinery_types:
        for breakdown in runs_breakdowns:
            pair = aligned_pair(
                runs,
                margins,
                runs_breakdown=breakdown,
                refinery_type=refinery,
                metric=metric,
                display_mode=display_mode,
            )
            base = {
                "refinery_type": refinery,
                "refinery_short": SHORT.get(refinery, refinery),
                "runs_breakdown": breakdown,
                "n_weeks": len(pair),
            }
            if len(pair) < min_overlap:
                rows.append(
                    {
                        **base,
                        "lag0_pearson": np.nan,
                        "best_lag": np.nan,
                        "best_pearson": np.nan,
                        "best_spearman": np.nan,
                        "delta_pearson": np.nan,
                        "r2_lag0": np.nan,
                        "r2_best": np.nan,
                    }
                )
                continue

            lag_df = lag_correlations(pair, max_lag, min_overlap)
            lag0 = lag_df.loc[lag_df["lag_weeks"] == 0]
            lag0_p = float(lag0["pearson"].iloc[0]) if not lag0.empty else np.nan
            best = best_lag_row(lag_df, "pearson")
            lagged = apply_lag(pair, int(best["lag_weeks"]))
            _, _, r2_0 = ols_fit(pair["margin"], pair["runs_kbd"])
            _, _, r2_b = ols_fit(lagged["margin_lagged"], lagged["runs_kbd"])
            rows.append(
                {
                    **base,
                    "lag0_pearson": lag0_p,
                    "best_lag": int(best["lag_weeks"]),
                    "best_pearson": float(best["pearson"]),
                    "best_spearman": float(best["spearman"]),
                    "delta_pearson": float(best["pearson"] - lag0_p)
                    if pd.notna(lag0_p)
                    else np.nan,
                    "r2_lag0": r2_0,
                    "r2_best": r2_b,
                }
            )
    return pd.DataFrame(rows)


print("Helpers ready.")
print("Example margin col:", margin_column(margins_wide, FOCUS_REFINERY, FOCUS_METRIC, FOCUS_DISPLAY))

Helpers ready.
Example margin col: Fueloriented_refineries_net_profit


## 1. Deep dive — focus pair

Overlay + lag correlation curve for the configured focus pair.

In [15]:
focus = aligned_pair(
    runs_wide,
    margins_wide,
    runs_breakdown=FOCUS_RUNS,
    refinery_type=FOCUS_REFINERY,
    metric=FOCUS_METRIC,
    display_mode=FOCUS_DISPLAY,
)
assert not focus.empty, "No overlapping weeks for focus pair — check config / cache"

ylab = margin_axis_label(FOCUS_METRIC, FOCUS_DISPLAY)
print(f"{FOCUS_RUNS} runs × {FOCUS_REFINERY} {FOCUS_METRIC} ({ylab})")
print(f"Overlapping weeks: {len(focus)}  ({focus['week_start'].min().date()} → {focus['week_start'].max().date()})")
display(focus.tail(3))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(
        x=focus["week_start"],
        y=focus["runs_kbd"],
        name=f"{FOCUS_RUNS} runs",
        line=dict(color="#58a6ff", width=2),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=focus["week_start"],
        y=focus["margin"],
        name=f"{SHORT.get(FOCUS_REFINERY, FOCUS_REFINERY)} {FOCUS_METRIC} margin",
        line=dict(color="#d2a8ff", width=2, dash="dot"),
    ),
    secondary_y=True,
)
fig.update_layout(
    title="Focus pair — runs vs margin (same week)",
    hovermode="x unified",
    template="plotly_white",
    height=420,
)
fig.update_yaxes(title_text="Runs (kb/d)", secondary_y=False)
fig.update_yaxes(title_text=f"Margin ({ylab})", secondary_y=True)
fig.show()

Independents runs × Fuel-oriented refineries net (RMB)
Overlapping weeks: 176  (2023-01-06 → 2026-07-17)


,week_start,processing_kt,runs_kbd,margin
173,2026-07-03,"3,651.590","3,823.736","769,774.810"
174,2026-07-10,"3,440.340","3,602.527","-1,647,057.750"
175,2026-07-17,"3,558.900","3,726.677","-2,585,776.340"


In [16]:
focus_lags = lag_correlations(focus, MAX_LAG_WEEKS, MIN_OVERLAP)
best = best_lag_row(focus_lags, "pearson")
lag0 = focus_lags.loc[focus_lags["lag_weeks"] == 0].iloc[0]

print(
    f"Lag 0:  pearson={lag0['pearson']:.3f}  spearman={lag0['spearman']:.3f}  n={int(lag0['n'])}"
)
print(
    f"Best (≥0): lag={int(best['lag_weeks'])}w  "
    f"pearson={best['pearson']:.3f}  spearman={best['spearman']:.3f}  n={int(best['n'])}"
)
print(f"Δ pearson (best − lag0): {best['pearson'] - lag0['pearson']:+.3f}")
display(focus_lags)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=focus_lags["lag_weeks"],
        y=focus_lags["pearson"],
        mode="lines+markers",
        name="Pearson",
        line=dict(color="#58a6ff", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=focus_lags["lag_weeks"],
        y=focus_lags["spearman"],
        mode="lines+markers",
        name="Spearman",
        line=dict(color="#f85149", width=2, dash="dot"),
    )
)
fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="#8b949e")
fig.add_vline(
    x=int(best["lag_weeks"]),
    line_width=1,
    line_dash="dot",
    line_color="#3fb950",
    annotation_text=f"best {int(best['lag_weeks'])}w",
)
fig.update_layout(
    title=f"Cross-correlation — {FOCUS_RUNS} runs vs {SHORT.get(FOCUS_REFINERY, FOCUS_REFINERY)} margin<br>"
    "<sup>Positive lag = margin leads runs</sup>",
    xaxis_title="Lag (weeks)",
    yaxis_title="Correlation",
    template="plotly_white",
    height=420,
)
fig.show()

Lag 0:  pearson=0.040  spearman=-0.071  n=176
Best (≥0): lag=8w  pearson=0.429  spearman=0.080  n=168
Δ pearson (best − lag0): +0.389


,lag_weeks,n,pearson,spearman
0,-4,172,-0.041,-0.089
1,-3,173,-0.021,-0.081
2,-2,174,-0.005,-0.074
3,-1,175,0.021,-0.068
4,0,176,0.040,-0.071
5,1,175,0.071,-0.065
6,2,174,0.145,-0.023
7,3,173,0.230,0.016
8,4,172,0.308,0.057
9,5,171,0.360,0.065


In [17]:
def scatter_with_fit(df: pd.DataFrame, xcol: str, title: str, xlab: str) -> go.Figure:
    slope, intercept, r2 = ols_fit(df[xcol], df["runs_kbd"])
    xline = np.linspace(df[xcol].min(), df[xcol].max(), 50)
    yline = intercept + slope * xline

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df[xcol],
            y=df["runs_kbd"],
            mode="markers",
            marker=dict(size=8, color="#58a6ff", opacity=0.7),
            text=[d.strftime("%Y-%m-%d") for d in df["week_start"]],
            hovertemplate="Week: %{text}<br>Margin: %{x:.1f}<br>Runs: %{y:,.0f}<extra></extra>",
            name="weeks",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=xline,
            y=yline,
            mode="lines",
            line=dict(color="#f85149", width=2),
            name=f"OLS  R²={r2:.3f}",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title=xlab,
        yaxis_title="Runs (kb/d)",
        template="plotly_white",
        height=420,
    )
    return fig, r2


best_lag = int(best["lag_weeks"])
lagged = apply_lag(focus, best_lag)
xlab = f"Margin ({ylab})"

fig0, r2_0 = scatter_with_fit(
    focus,
    "margin",
    f"Scatter lag 0 — {FOCUS_RUNS} vs {SHORT.get(FOCUS_REFINERY, FOCUS_REFINERY)}",
    xlab,
)
fig0.show()

figb, r2_b = scatter_with_fit(
    lagged,
    "margin_lagged",
    f"Scatter lag {best_lag}w — margin leads runs",
    f"Margin {best_lag}w earlier ({ylab})",
)
figb.show()
print(f"R² lag0={r2_0:.3f}  →  R² lag{best_lag}w={r2_b:.3f}  (Δ {r2_b - r2_0:+.3f})")

R² lag0=0.002  →  R² lag8w=0.184  (Δ +0.182)


### Runs vs margin — best-lag overlay

Same dual-axis chart as the dashboard, but the margin series is shifted forward by the best lag so it sits on the week when runs are expected to react. Side-by-side with lag 0 for visual comparison.

In [18]:
def overlay_runs_margin(
    df: pd.DataFrame,
    *,
    margin_col: str,
    title: str,
    margin_name: str,
    ylab: str,
) -> go.Figure:
    """Dual-axis runs vs margin; margin_col is 'margin' or 'margin_lagged'."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=df["week_start"],
            y=df["runs_kbd"],
            name=f"{FOCUS_RUNS} runs",
            line=dict(color="#58a6ff", width=2),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=df["week_start"],
            y=df[margin_col],
            name=margin_name,
            line=dict(color="#d2a8ff", width=2, dash="dot"),
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title=title,
        hovermode="x unified",
        template="plotly_white",
        height=420,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.update_yaxes(title_text="Runs (kb/d)", secondary_y=False)
    fig.update_yaxes(title_text=f"Margin ({ylab})", secondary_y=True)
    return fig


short = SHORT.get(FOCUS_REFINERY, FOCUS_REFINERY)

# Lag 0: both series on their native week
overlay_runs_margin(
    focus,
    margin_col="margin",
    title=f"Runs vs margin — lag 0 (same week)<br><sup>{FOCUS_RUNS} × {short}</sup>",
    margin_name=f"{short} {FOCUS_METRIC} (same week)",
    ylab=ylab,
).show()

# Best lag: margin from t−L plotted on week t (when runs react)
overlay_runs_margin(
    lagged,
    margin_col="margin_lagged",
    title=(
        f"Runs vs margin — lag {best_lag}w (margin leads)"
        f"<br><sup>Margin shifted forward {best_lag} weeks onto the runs date</sup>"
    ),
    margin_name=f"{short} {FOCUS_METRIC} ({best_lag}w earlier)",
    ylab=ylab,
).show()

print(
    f"Aligned weeks after lag {best_lag}w: {len(lagged)}  "
    f"({lagged['week_start'].min().date()} → {lagged['week_start'].max().date()})"
)

Aligned weeks after lag 8w: 168  (2023-03-10 → 2026-07-17)


## 2. Who reacts? — margin models × runs groups

For each **profit model** (Fuel / Chem / Aromatics) and each **runs group**
(Independents / SOEs / National), find the lag that maximises |Pearson|
among lags 0…`MAX_LAG_WEEKS`.

Read the tables as:
- **best_lag** — how many weeks later runs tend to move
- **best_pearson / delta_pearson** — strength vs same-week fit
- **r2_best** — linear explanatory power after lagging

In [19]:
grid = sweep_grid(
    runs_wide,
    margins_wide,
    REFINERY_TYPES,
    RUNS_BREAKDOWNS,
    metric=COMPARE_METRIC,
    display_mode=COMPARE_DISPLAY,
    max_lag=MAX_LAG_WEEKS,
    min_overlap=MIN_OVERLAP,
)

print(
    f"Metric: {COMPARE_METRIC} ({margin_axis_label(COMPARE_METRIC, COMPARE_DISPLAY)})  |  "
    f"max lag {MAX_LAG_WEEKS}w"
)
display(
    grid.sort_values(["runs_breakdown", "refinery_short"]).reset_index(drop=True)
)

Metric: net (RMB)  |  max lag 12w


,refinery_type,refinery_short,runs_breakdown,n_weeks,lag0_pearson,best_lag,best_pearson,best_spearman,delta_pearson,r2_lag0,r2_best
0,Oil Refining and Aromatics Complexes,Aromatics,Independents,176,-0.293,0,-0.293,-0.189,0.000,0.086,0.086
1,Oil Refining and Chemical Complexes,Chem,Independents,176,-0.221,0,-0.221,-0.189,0.000,0.049,0.049
2,Fuel-oriented refineries,Fuel,Independents,176,0.040,8,0.429,0.080,0.389,0.002,0.184
3,Oil Refining and Aromatics Complexes,Aromatics,National,176,-0.323,0,-0.323,-0.323,0.000,0.104,0.104
4,Oil Refining and Chemical Complexes,Chem,National,176,-0.185,9,0.300,0.120,0.485,0.034,0.090
5,Fuel-oriented refineries,Fuel,National,176,0.157,9,0.541,0.304,0.384,0.025,0.292
6,Oil Refining and Aromatics Complexes,Aromatics,SOEs,176,-0.287,11,0.304,0.226,0.590,0.082,0.092
7,Oil Refining and Chemical Complexes,Chem,SOEs,176,-0.135,9,0.322,0.224,0.458,0.018,0.104
8,Fuel-oriented refineries,Fuel,SOEs,176,0.196,9,0.527,0.395,0.331,0.038,0.277


In [20]:
def heatmap(df: pd.DataFrame, value: str, title: str, zfmt: str = ".2f") -> go.Figure:
    pivot = df.pivot(index="runs_breakdown", columns="refinery_short", values=value)
    # Stable axis order
    pivot = pivot.reindex(index=RUNS_BREAKDOWNS, columns=[SHORT[r] for r in REFINERY_TYPES])
    fig = px.imshow(
        pivot,
        text_auto=zfmt,
        aspect="auto",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0 if value != "best_lag" else None,
        title=title,
    )
    if value == "best_lag":
        fig.update_traces(zmid=None)
        fig.update_layout(coloraxis_colorbar_title="weeks")
        fig.update_coloraxes(colorscale="Viridis")
    else:
        fig.update_layout(coloraxis_colorbar_title="corr")
    fig.update_layout(template="plotly_white", height=380, xaxis_title="Margin model", yaxis_title="Runs group")
    return fig


heatmap(grid, "lag0_pearson", "Same-week Pearson (lag 0)").show()
heatmap(grid, "best_pearson", "Best-lag Pearson (margin leads)").show()
heatmap(grid, "best_lag", "Best lag (weeks)", zfmt=".0f").show()
heatmap(grid, "delta_pearson", "Δ Pearson (best − lag0)").show()

## 3. Lag curves by player

Hold the margin model fixed and overlay Independents / SOEs / National correlation curves.
Useful to see whether teapots reprice throughput faster than SOEs.

In [21]:
PLAYER_COLORS = {
    "Independents": "#f85149",
    "SOEs": "#58a6ff",
    "National": "#3fb950",
}


def plot_player_lag_curves(
    refinery_type: str,
    *,
    metric: str = COMPARE_METRIC,
    display_mode: str = COMPARE_DISPLAY,
) -> go.Figure:
    fig = go.Figure()
    for breakdown in RUNS_BREAKDOWNS:
        pair = aligned_pair(
            runs_wide,
            margins_wide,
            runs_breakdown=breakdown,
            refinery_type=refinery_type,
            metric=metric,
            display_mode=display_mode,
        )
        if pair.empty:
            continue
        lags = lag_correlations(pair, MAX_LAG_WEEKS, MIN_OVERLAP)
        fig.add_trace(
            go.Scatter(
                x=lags["lag_weeks"],
                y=lags["pearson"],
                mode="lines+markers",
                name=breakdown,
                line=dict(color=PLAYER_COLORS.get(breakdown, "#8b949e"), width=2),
            )
        )
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="#8b949e")
    fig.update_layout(
        title=f"Lag curves by runs group — signal: {SHORT.get(refinery_type, refinery_type)} "
        f"{metric} ({margin_axis_label(metric, display_mode)})",
        xaxis_title="Lag (weeks) — positive = margin leads",
        yaxis_title="Pearson",
        template="plotly_white",
        height=420,
    )
    return fig


for refinery in REFINERY_TYPES:
    plot_player_lag_curves(refinery).show()

## 4. Same player, different economic signals

Hold the runs group fixed and compare which profit model (Fuel / Chem / Aromatics)
is the stronger lead indicator for that group's runs.

In [22]:
SIGNAL_COLORS = {
    "Fuel": "#e3b341",
    "Chem": "#d2a8ff",
    "Aromatics": "#79c0ff",
}


def plot_signal_lag_curves(
    runs_breakdown: str,
    *,
    metric: str = COMPARE_METRIC,
    display_mode: str = COMPARE_DISPLAY,
) -> go.Figure:
    fig = go.Figure()
    for refinery in REFINERY_TYPES:
        short = SHORT.get(refinery, refinery)
        pair = aligned_pair(
            runs_wide,
            margins_wide,
            runs_breakdown=runs_breakdown,
            refinery_type=refinery,
            metric=metric,
            display_mode=display_mode,
        )
        if pair.empty:
            continue
        lags = lag_correlations(pair, MAX_LAG_WEEKS, MIN_OVERLAP)
        fig.add_trace(
            go.Scatter(
                x=lags["lag_weeks"],
                y=lags["pearson"],
                mode="lines+markers",
                name=short,
                line=dict(color=SIGNAL_COLORS.get(short, "#8b949e"), width=2),
            )
        )
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="#8b949e")
    fig.update_layout(
        title=f"Lag curves by margin model — runs: {runs_breakdown} "
        f"({metric}, {margin_axis_label(metric, display_mode)})",
        xaxis_title="Lag (weeks) — positive = margin leads",
        yaxis_title="Pearson",
        template="plotly_white",
        height=420,
    )
    return fig


for breakdown in RUNS_BREAKDOWNS:
    plot_signal_lag_curves(breakdown).show()

## 5. Insight summary

Auto-ranked takeaways from the comparison grid. Treat these as exploratory —
correlation ≠ causation, and weekly SCI models are themselves modelled.

In [23]:
g = grid.dropna(subset=["best_pearson"]).copy()
if g.empty:
    print("No valid pairs — widen cache history or lower MIN_OVERLAP.")
else:
    # Strongest lagged relationships
    top = g.reindex(g["best_pearson"].abs().sort_values(ascending=False).index).head(5)
    print("Strongest |Pearson| after best lag")
    display(
        top[
            [
                "runs_breakdown",
                "refinery_short",
                "best_lag",
                "lag0_pearson",
                "best_pearson",
                "delta_pearson",
                "r2_best",
            ]
        ].reset_index(drop=True)
    )

    # Biggest improvement from lagging (supports the hypothesis most)
    improved = g.reindex(g["delta_pearson"].abs().sort_values(ascending=False).index).head(5)
    print("Largest |Δ Pearson| from introducing a lag")
    display(
        improved[
            [
                "runs_breakdown",
                "refinery_short",
                "best_lag",
                "lag0_pearson",
                "best_pearson",
                "delta_pearson",
            ]
        ].reset_index(drop=True)
    )

    # Typical reaction speed by player (mean best lag across margin models)
    speed = (
        g.groupby("runs_breakdown", sort=False)[["best_lag", "best_pearson", "delta_pearson"]]
        .mean()
        .reindex(RUNS_BREAKDOWNS)
    )
    print("Average best lag / corr by runs group (across margin models)")
    display(speed)

    # Which signal is most informative for Independents?
    indep = g[g["runs_breakdown"] == "Independents"].sort_values(
        "best_pearson", key=lambda s: s.abs(), ascending=False
    )
    if not indep.empty:
        print("Independents — ranking of margin signals by |best Pearson|")
        display(
            indep[
                ["refinery_short", "best_lag", "lag0_pearson", "best_pearson", "delta_pearson", "r2_best"]
            ].reset_index(drop=True)
        )

    # Flip COMPARE_METRIC / COMPARE_DISPLAY above and re-run from Config to stress-test
    print(
        "Tip: re-run with COMPARE_METRIC in {'gross','net','contribution'} and "
        "COMPARE_DISPLAY in {'pct','money'} to see if the lag story is metric-dependent."
    )

Strongest |Pearson| after best lag


,runs_breakdown,refinery_short,best_lag,lag0_pearson,best_pearson,delta_pearson,r2_best
0,National,Fuel,9,0.157,0.541,0.384,0.292
1,SOEs,Fuel,9,0.196,0.527,0.331,0.277
2,Independents,Fuel,8,0.040,0.429,0.389,0.184
3,National,Aromatics,0,-0.323,-0.323,0.000,0.104
4,SOEs,Chem,9,-0.135,0.322,0.458,0.104


Largest |Δ Pearson| from introducing a lag


,runs_breakdown,refinery_short,best_lag,lag0_pearson,best_pearson,delta_pearson
0,SOEs,Aromatics,11,-0.287,0.304,0.590
1,National,Chem,9,-0.185,0.300,0.485
2,SOEs,Chem,9,-0.135,0.322,0.458
3,Independents,Fuel,8,0.040,0.429,0.389
4,National,Fuel,9,0.157,0.541,0.384


Average best lag / corr by runs group (across margin models)


,best_lag,best_pearson,delta_pearson
runs_breakdown,,,
Independents,2.667,-0.028,0.130
SOEs,9.667,0.384,0.460
National,6.000,0.173,0.290


Independents — ranking of margin signals by |best Pearson|


,refinery_short,best_lag,lag0_pearson,best_pearson,delta_pearson,r2_best
0,Fuel,8,0.040,0.429,0.389,0.184
1,Aromatics,0,-0.293,-0.293,0.000,0.086
2,Chem,0,-0.221,-0.221,0.000,0.049


Tip: re-run with COMPARE_METRIC in {'gross','net','contribution'} and COMPARE_DISPLAY in {'pct','money'} to see if the lag story is metric-dependent.


## Next steps (after reviewing results)

If a stable lag shows up (e.g. Independents × Fuel net, 3–6 weeks):

1. Add an optional **lag (weeks)** control on the Economics tab scatter
2. Show lag-0 vs lagged R² in a small annotation
3. Optionally default the scatter to the empirically best lag for the selected pair

Keep this notebook as the lab — only promote into `app.py` once the lag looks robust across metrics.